# TimesFM vs NeuralProphet benchmark

Цель: сравнить время и качество прогноза `TimesFM` и `NeuralProphet` на 2-3 рядах из БД.

## Как запускать
1. Выберите `RUN_MODE = 'cpu'` или `RUN_MODE = 'gpu'` в ячейке конфигурации.
2. Запустите ноутбук полностью.
3. Перезапустить среду (при работе в колаб лучше сменить среду выполнения)
3. Повторите запуск для второго режима (`cpu`/`gpu`) и сравните итоговые таблицы.

| model | mode | series_id | n_train | horizon | elapsed_sec | mape |
|---|---|---|---:|---:|---:|---:|
| TimesFM | gpu | Индейка\|крыло\|Москва | 981 | 13 | 8.64 | 2.78 |
| NeuralProphet | gpu | Индейка\|крыло\|Москва | 981 | 13 | 4.64 | 13.70 |
| TimesFM | gpu | Индейка\|филе бедра\|Москва | 981 | 13 | 2.56 | 7.76 |
| NeuralProphet | gpu | Индейка\|филе бедра\|Москва | 981 | 13 | 7.06 | 43.54 |
| TimesFM | gpu | Кура\|бедро\|Москва | 981 | 13 | 2.09 | 3.39 |
| NeuralProphet | gpu | Кура\|бедро\|Москва | 981 | 13 | 4.57 | 13.48 |
| TimesFM | cpu | Индейка\|крыло\|Москва | 981 | 13 | 23.93 | 2.78 |
| NeuralProphet | cpu | Индейка\|крыло\|Москва | 981 | 13 | 5.05 | 15.34 |
| TimesFM | cpu | Индейка\|филе бедра\|Москва | 981 | 13 | 3.18 | 7.76 |
| NeuralProphet | cpu | Индейка\|филе бедра\|Москва | 981 | 13 | 5.28 | 10.73 |
| TimesFM | cpu | Кура\|бедро\|Москва | 981 | 13 | 5.38 | 3.39 |
| NeuralProphet | cpu | Кура\|бедро\|Москва | 981 | 13 | 6.11 | 3.57 |

In [ ]:
# Раскомментируйте при первом запуске (например в Colab)
!pip install -q pymysql
!pip install -q --no-deps git+https://github.com/google-research/timesfm.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip install neuralprophet -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.4/145.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incomp

In [ ]:
import os

# Запустите ноутбук отдельно для cpu и gpu режимов
RUN_MODE = "cpu"  # "cpu" | "gpu"
if RUN_MODE not in {"cpu", "gpu"}:
    raise ValueError("RUN_MODE must be 'cpu' or 'gpu'")

if RUN_MODE == "cpu":
    # Отключаем GPU для текущего запуска
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

import time
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import pymysql
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error
from neuralprophet import NeuralProphet, set_log_level
import timesfm

set_log_level("ERROR")
print("RUN_MODE:", RUN_MODE)
print("torch.cuda.is_available():", torch.cuda.is_available())

ERROR:NP.plotly:Importing plotly failed. Interactive plots will not work.
ERROR:NP.plotly:Importing plotly failed. Interactive plots will not work.


RUN_MODE: cpu
torch.cuda.is_available(): False


In [ ]:
@dataclass
class BenchmarkConfig:
    n_test: int = 13
    n_series: int = 3
    min_history_points: int = 120
    freq: str = "W-MON"
    np_epochs: int = 30


cfg = BenchmarkConfig()

# Подставьте доступ к вашей БД
db_config = {
    'host': 'IP',
    'port': 3308,
    'user': 'user',
    'password': 'password',
    'database': 'komida'
}

# SQL можно адаптировать под вашу схему.
# Ниже пример для мясных мониторингов с агрегацией до недельной частоты.
def load_weekly_series_from_db(config: Dict, limit_series: int, min_points: int) -> pd.DataFrame:
    query = f"""
    WITH base AS (
        SELECT
            CONCAT(product, '|', IFNULL(product_type, 'null'), '|', IFNULL(federal_okrug, 'all')) AS series_id,
            DATE(date) AS ds,
            AVG(price) AS y
        FROM raw_monitorings_meat
        WHERE price IS NOT NULL
          AND is_outlier = 0
        GROUP BY 1, 2
    ),
    weekly AS (
        SELECT
            series_id,
            DATE_SUB(ds, INTERVAL WEEKDAY(ds) DAY) AS week_start,
            AVG(y) AS y
        FROM base
        GROUP BY 1, 2
    ),
    candidates AS (
        SELECT series_id
        FROM weekly
        GROUP BY series_id
        HAVING COUNT(*) >= {min_points}
        ORDER BY COUNT(*) DESC
        LIMIT {limit_series}
    )
    SELECT w.series_id AS ID, w.week_start AS ds, w.y
    FROM weekly w
    JOIN candidates c ON c.series_id = w.series_id
    ORDER BY ID, ds
    """

    conn = pymysql.connect(**config)
    try:
        df = pd.read_sql(query, conn)
    finally:
        conn.close()

    df["ds"] = pd.to_datetime(df["ds"])
    df = df.dropna(subset=["y"]).sort_values(["ID", "ds"]).reset_index(drop=True)
    return df


df_all = load_weekly_series_from_db(db_config, cfg.n_series, cfg.min_history_points)
print(df_all.groupby("ID").size())
df_all.head()

WARNING - (py.warnings._showwarnmsg) - /tmp/ipykernel_12272/1616545929.py:59: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)

  df = pd.read_sql(query, conn)



ID
Индейка|крыло|Москва         994
Индейка|филе бедра|Москва    994
Кура|бедро|Москва            994
dtype: int64


,ID,ds,y
0,Индейка|крыло|Москва,2005-10-31,99.000000
1,Индейка|крыло|Москва,2005-12-05,61.710001
2,Индейка|крыло|Москва,2005-12-12,70.694000
3,Индейка|крыло|Москва,2006-01-16,68.078334
4,Индейка|крыло|Москва,2006-02-27,65.497273


In [ ]:
def safe_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true != 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def split_train_test(series_df: pd.DataFrame, n_test: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    series_df = series_df.sort_values("ds").copy()
    train = series_df.iloc[:-n_test].copy()
    test = series_df.iloc[-n_test:].copy()
    if len(train) < 10 or len(test) < 1:
        raise ValueError("Not enough points for train/test split")
    return train, test


def run_timesfm(train: pd.DataFrame, horizon: int) -> np.ndarray:
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    model.compile(
        timesfm.ForecastConfig(
            max_context=min(1024, len(train)),
            max_horizon=horizon,
            normalize_inputs=True,
            use_continuous_quantile_head=True,
            infer_is_positive=True,
        )
    )
    inputs = train["y"].values.astype(np.float32)
    point_forecast, _ = model.forecast(horizon=horizon, inputs=[inputs])
    return point_forecast[0][:horizon]


def run_neuralprophet(train: pd.DataFrame, horizon: int, freq: str, epochs: int) -> np.ndarray:
    accelerator = "gpu" if (RUN_MODE == "gpu" and torch.cuda.is_available()) else "cpu"
    model = NeuralProphet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        n_lags=min(13, max(2, len(train) // 4)),
        n_forecasts=horizon,
        learning_rate=1e-3,
        accelerator=accelerator,
        trainer_config={"accelerator": accelerator},
    )
    model.fit(train[["ds", "y"]], freq=freq, minimal=True, epochs=epochs)
    future = model.make_future_dataframe(train[["ds", "y"]], n_historic_predictions=True, periods=horizon)
    forecast = model.predict(future)
    preds = forecast[forecast["y"].isna()].copy()
    yhat_cols = [f"yhat{i}" for i in range(1, horizon + 1)]
    preds.loc[:, "pred"] = preds[yhat_cols].max(axis=1)
    return preds["pred"].values[:horizon]


def benchmark_one_series(series_df: pd.DataFrame, config: BenchmarkConfig) -> List[Dict]:
    train, test = split_train_test(series_df, config.n_test)

    rows = []

    t0 = time.perf_counter()
    preds_tf = run_timesfm(train, config.n_test)
    tf_sec = time.perf_counter() - t0
    rows.append(
        {
            "model": "TimesFM",
            "mode": RUN_MODE,
            "series_id": series_df["ID"].iloc[0],
            "n_train": len(train),
            "horizon": config.n_test,
            "elapsed_sec": round(tf_sec, 2),
            "mape": round(safe_mape(test["y"].values, preds_tf), 2),
        }
    )

    t0 = time.perf_counter()
    preds_np = run_neuralprophet(train, config.n_test, config.freq, config.np_epochs)
    np_sec = time.perf_counter() - t0
    rows.append(
        {
            "model": "NeuralProphet",
            "mode": RUN_MODE,
            "series_id": series_df["ID"].iloc[0],
            "n_train": len(train),
            "horizon": config.n_test,
            "elapsed_sec": round(np_sec, 2),
            "mape": round(safe_mape(test["y"].values, preds_np), 2),
        }
    )

    return rows

In [ ]:
all_rows = []
for sid, df_sid in df_all.groupby("ID"):
    try:
        rows = benchmark_one_series(df_sid, cfg)
        all_rows.extend(rows)
        print(f"done: {sid}")
    except Exception as exc:
        print(f"skip {sid}: {exc}")

results = pd.DataFrame(all_rows)
results


In [ ]:
if results.empty:
    print("No benchmark rows. Check DB config/query.")
else:
    summary = (
        results
        .groupby(["model", "mode"], as_index=False)
        .agg(
            series_count=("series_id", "nunique"),
            elapsed_sec_mean=("elapsed_sec", "mean"),
            elapsed_sec_median=("elapsed_sec", "median"),
            mape_mean=("mape", "mean"),
        )
        .sort_values(["mode", "elapsed_sec_median"])
    )
    display(summary)

    pivot = results.pivot_table(index="series_id", columns="model", values="elapsed_sec")
    if {"TimesFM", "NeuralProphet"}.issubset(pivot.columns):
        pivot["speedup_timesfm_vs_np"] = pivot["NeuralProphet"] / pivot["TimesFM"]
        display(pivot)

    results.to_csv(f"benchmark_{RUN_MODE}.csv", index=False)
    print(f"Saved benchmark_{RUN_MODE}.csv")

,model,mode,series_count,elapsed_sec_mean,elapsed_sec_median,mape_mean
0,NeuralProphet,cpu,3,5.48,5.28,9.880000
1,TimesFM,cpu,3,10.83,5.38,4.643333


model,NeuralProphet,TimesFM,speedup_timesfm_vs_np
series_id,,,
Индейка|крыло|Москва,5.05,23.93,0.211032
Индейка|филе бедра|Москва,5.28,3.18,1.660377
Кура|бедро|Москва,6.11,5.38,1.135688


Saved benchmark_cpu.csv
